## 1. Important Imports

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd 
import numpy as np 
import seaborn as se 
import matplotlib as plt
import random 

## 2. Createting PySpark Session

In [0]:
spark = SparkSession.builder.appName('Data_Analysis_with_PySpark').getOrCreate()

## 3. Generating Data

In [0]:
names = [
    "Alice", "Bob", "Charlie", "David", "Eve", "Fiona", "George", "Hannah",
    "Ivy", "Jack", "Kaitlyn", "Liam", "Olivia", "Liam", "Emma", "Noah", 
    "Ava", "Oliver", "Charlotte", "Elijah", "Sophia", "James", "Amelia", 
    "Benjamin", "Isabella", "Lucas", "Mia", "Mason", "Harper", "Ethan", 
    "Evelyn", "Alexander", "Abigail", "Henry", "Ella", "Jackson", "Scarlett", 
    "Aiden", "Grace", "Samuel", "Lily", "Sebastian"
]
genders = ["Male", "Female", None]
subjects = ["Math", "Science", "History", "English", "Art", "PE", None]
cities = [
    "New York", "Los Angeles", "Chicago", "Houston", 
    "Bangalore", "Hajipur", "Sitamardhi", "MP", None
]
states = ["NY", "CA", "IL", "TX", "Bihar", "Karnataka", "Sitamardhi", None]
countries = ["USA", "India", "Pakistan", "Nepal", "China", None]
graduated_status = ["Yes", "No", None]

data = [
    (
        i, 
        random.choice(names),  # student_name
        random.choice([random.randint(18, 25), None]),  # age
        random.choice(genders),  # gender
        random.choice(subjects),  # subject
        random.choice([random.randint(50, 100), None]),  # marks
        random.choice(cities),  # city
        random.choice(states),  # state
        random.choice(countries),  # country
        random.choice(graduated_status),  # graduated
    )
    for i in range(1, 501)
]

## 4. Creating a DATAFRAME

In [0]:
schema = StructType([
    StructField("student_id", IntegerType(), True),
    StructField("student_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("marks", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("graduated", StringType(), True),
])

df = spark.createDataFrame(data,schema=schema)
df.show(5)

+----------+------------+----+------+-------+-----+---------+---------+--------+---------+
|student_id|student_name| age|gender|subject|marks|     city|    state| country|graduated|
+----------+------------+----+------+-------+-----+---------+---------+--------+---------+
|         1|       Fiona|  19|Female|English|   83|       MP|       CA|   China|       No|
|         2|         Mia|  20|  null|History|   87|Bangalore|       TX|Pakistan|       No|
|         3|       Aiden|  18|  Male|History| null|  Chicago|       CA|    null|       No|
|         4|        Liam|null|  Male|    Art|   63| New York|Karnataka|Pakistan|     null|
|         5|        Noah|  25|Female|Science| null|  Chicago|    Bihar|   India|       No|
+----------+------------+----+------+-------+-----+---------+---------+--------+---------+
only showing top 5 rows



## 5.OverView of DataFrame

In [0]:
df.show(10)

+----------+------------+----+------+-------+-----+---------+----------+--------+---------+
|student_id|student_name| age|gender|subject|marks|     city|     state| country|graduated|
+----------+------------+----+------+-------+-----+---------+----------+--------+---------+
|         1|       Fiona|  19|Female|English|   83|       MP|        CA|   China|       No|
|         2|         Mia|  20|  null|History|   87|Bangalore|        TX|Pakistan|       No|
|         3|       Aiden|  18|  Male|History| null|  Chicago|        CA|    null|       No|
|         4|        Liam|null|  Male|    Art|   63| New York| Karnataka|Pakistan|     null|
|         5|        Noah|  25|Female|Science| null|  Chicago|     Bihar|   India|       No|
|         6|     Charlie|  20|  Male|English| null|       MP|        IL|Pakistan|      Yes|
|         7|        Ella|null|Female|    Art|   51|Bangalore|        IL|   Nepal|      Yes|
|         8|         Bob|  23|Female|    Art|   75| New York|        IL|Pakistan

In [0]:
df.describe().show()

+-------+-----------------+------------+------------------+------+-------+------------------+----------+-----+-------+---------+
|summary|       student_id|student_name|               age|gender|subject|             marks|      city|state|country|graduated|
+-------+-----------------+------------+------------------+------+-------+------------------+----------+-----+-------+---------+
|  count|              500|         500|               248|   345|    423|               252|       465|  445|    422|      354|
|   mean|            250.5|        null| 21.31451612903226|  null|   null| 74.30952380952381|      null| null|   null|     null|
| stddev|144.4818327679989|        null|2.3632371111387656|  null|   null|15.081697456808824|      null| null|   null|     null|
|    min|                1|     Abigail|                18|Female|    Art|                50| Bangalore|Bihar|  China|       No|
|    max|              500|      Sophia|                25|  Male|Science|               100|Sita

In [0]:
df.dtypes

Out[7]: [('student_id', 'int'),
 ('student_name', 'string'),
 ('age', 'int'),
 ('gender', 'string'),
 ('subject', 'string'),
 ('marks', 'int'),
 ('city', 'string'),
 ('state', 'string'),
 ('country', 'string'),
 ('graduated', 'string')]

In [0]:
df.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- student_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- subject: string (nullable = true)
 |-- marks: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- graduated: string (nullable = true)



In [0]:
## Checking Missing Values
str_column = ['student_name','gender','subject','city','state','country','graduated']
num_column = ['student_id','age','marks']

missing_val_dict = {}
for index,column in enumerate(df.columns):
    if column in str_column:
        missing_str_count = df.filter(col(column).eqNullSafe(None)\
            | col(column).isNull()
            ).count()
        missing_val_dict.update({column:missing_str_count})
    if column in num_column:
        missing_num_count = df.where(col(column).isin([0,None,np.nan])).count()
        missing_val_dict.update({column:missing_num_count})
missing_df = pd.DataFrame.from_dict([missing_val_dict])
missing_df




,student_id,student_name,age,gender,subject,marks,city,state,country,graduated
0,0,0,0,155,77,0,35,55,78,146


## 6. Replacing Null Values

In [0]:
df = df.fillna({
    'gender' : 'UniSex',
    'subject' : 'Hindi',
    'city' : 'Kalitand',
    'State' : 'Others',
    'Country' : 'India',
    'graduated' : 'Failed'
})

In [0]:
## Checking Missing Values
str_column = ['student_name','gender','subject','city','state','country','graduated']
num_column = ['student_id','age','marks']

missing_val_dict = {}
for index,column in enumerate(df.columns):
    if column in str_column:
        missing_str_count = df.filter(col(column).eqNullSafe(None)\
            | col(column).isNull()
            ).count()
        missing_val_dict.update({column:missing_str_count})
    if column in num_column:
        missing_num_count = df.where(col(column).isin([0,None,np.nan])).count()
        missing_val_dict.update({column:missing_num_count})
missing_df = pd.DataFrame.from_dict([missing_val_dict])
missing_df




,student_id,student_name,age,gender,subject,marks,city,state,country,graduated
0,0,0,0,0,0,0,0,0,0,0


## 7. Checking duplicate Values in each column

In [0]:
for index,column in enumerate(df.columns):
    print(f"checking duplicates in the column: {column}")
    duplicate_count = df.groupBy(column).count().filter("count > 1 ")
    duplicate_count.show()


checking duplicates in the column: student_id
+----------+-----+
|student_id|count|
+----------+-----+
+----------+-----+

checking duplicates in the column: student_name
+------------+-----+
|student_name|count|
+------------+-----+
|       Lucas|   18|
|       Grace|   12|
|         Ivy|    9|
|    Isabella|   12|
|       James|   15|
|    Benjamin|   13|
|      Hannah|    7|
|        Ella|   13|
|      Evelyn|   12|
|        Noah|   14|
|       Mason|   14|
|       Ethan|   11|
|     Charlie|   12|
|         Mia|    9|
|         Bob|    9|
|        Liam|   25|
|     Jackson|    8|
|   Sebastian|   10|
|      Elijah|   14|
|      Samuel|   18|
+------------+-----+
only showing top 20 rows

checking duplicates in the column: age
+----+-----+
| age|count|
+----+-----+
|  22|   27|
|null|  252|
|  20|   44|
|  19|   25|
|  23|   31|
|  25|   35|
|  24|   21|
|  21|   25|
|  18|   40|
+----+-----+

checking duplicates in the column: gender
+------+-----+
|gender|count|
+------+-----+
|Fe

In [0]:
duplicate_counts = []
# Loop through each column in the DataFrame
for column in df.columns:
    # Group by the column and count duplicates (count > 1)
    dup_count = df.groupBy(column).count().filter("count > 1")
    # Get the number of duplicate values
    count = dup_count.count()
    # Append the result as a tuple (column name, duplicate count)
    duplicate_counts.append((column, count))
# Convert the list to a Pandas DataFrame
dup_df = pd.DataFrame(duplicate_counts, columns=['Column_Name', "Dup_count"])
# Show the Pandas DataFrame
dup_df


,Column_Name,Dup_count
0,student_id,0
1,student_name,41
2,age,9
3,gender,3
4,subject,7
5,marks,50
6,city,9
7,state,8
8,country,5
9,graduated,3


In [0]:
# Loop through each column in the DataFrame
for column in df.columns:
    print(column)
    dup_count = df.groupBy(column).count().filter("count > 1")
    # Get the number of duplicate values
    count = dup_count.count()
    # Append the result as a tuple (column name, duplicate count)
    duplicate_counts.append((column, count))
# Convert the list to a Pandas DataFrame
dup_df = pd.DataFrame(duplicate_counts, columns=['Column_Name', "Dup_count"])
# Show the Pandas DataFrame
dup_df


## 8.Descriptive Statistics and Basic Summarization

In [0]:
df.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- student_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = false)
 |-- subject: string (nullable = false)
 |-- marks: integer (nullable = true)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- country: string (nullable = false)
 |-- graduated: string (nullable = false)



In [0]:
# What are the central tendencies (mean, median, mode) of marks and age?
df.groupBy.("subject").agg(
    mean("marks").alias("marks_mean"),
    median("marks").alias("marks_median"),
    mean("age").alias("mean_age")
).show()



---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-4380487821440178>:1
----> 1 df.info

File /databricks/spark/python/pyspark/instrumentation_utils.py:48, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     46 start = time.perf_counter()
     47 try:
---> 48     res = func(*args, **kwargs)
     49     logger.log_success(
     50         module_name, class_name, function_name, time.perf_counter() - start, signature
     51     )
     52     return res

File /databricks/spark/python/pyspark/sql/dataframe.py:2964, in DataFrame.__getattr__(self, name)
   2934 """Returns the :class:`Column` denoted by ``name``.
   2935 
   2936 .. versionadded:: 1.3.0
   (...)
   2961 +---+
   2962 """
   2963 if name not in self.columns:
-> 2964     raise AttributeError(
   2965         "'%s' object has no attribute '%s'" % (self.__class__.__name__, name)
   2966     )
   2967 jc = self._